In [1]:
import sys
from pathlib import Path

# Find project root containing "src"
CWD = Path.cwd()
candidates = [CWD, CWD.parent, CWD.parent.parent, CWD.parent.parent.parent]
PROJECT_ROOT = None
for p in candidates:
    if (p / "src").exists():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT is None:
    raise RuntimeError(f"Could not find project root containing 'src' from cwd={CWD}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("✅ PROJECT_ROOT added to sys.path:", PROJECT_ROOT)

✅ PROJECT_ROOT added to sys.path: /home/johnathanc/APM/LIO


In [2]:
import json
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional, Dict, Any, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dateutil import parser as dtparser
import ipywidgets as widgets
from IPython.display import display, clear_output

In [3]:
from src.apm_core.ssot import get_sensor
from src.apm_core.raw_pull import build_wide_frame

In [4]:
import json
import logging
from pathlib import Path

from src.apm_core.settings import load_ini
from src.apm_core.db_interface import DBInterface

SITE_ROOT = PROJECT_ROOT
CONFIG_JSON_PATH = SITE_ROOT / "etc" / "config.json"

# Logger
logger = logging.getLogger("training_notebook")
if not logger.handlers:
    logger.setLevel(logging.INFO)
    h = logging.StreamHandler()
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(h)

# SSOT
with open(CONFIG_JSON_PATH, "r") as f:
    ssot = json.load(f)
SENSORS = [k for k in ssot.keys() if k not in ("site", "debug", "timezone")]
print("✅ Loaded SSOT sensors:", SENSORS)

# Ini + DB (IMPORTANT: pass SITE_ROOT Path)
settings = load_ini(SITE_ROOT, debug=False)
db = DBInterface(settings=settings, logger=logger)

print("✅ DB ready")

2026-02-26 16:49:26,100 | INFO | PostgreSQL pool initialised
2026-02-26 16:49:26,120 | INFO | MSSQL connected | table=APM_Scalability


✅ Loaded SSOT sensors: ['Mill1PinionConditionScore', 'Mill1TrunnionConditionScore', 'Mill1MotorConditionScore', 'Mill1GearboxConditionScore']
✅ DB ready


In [5]:
def interval_string_from_other(other: Dict[str, Any]) -> str:
    n = int(other.get("granularity", 15))
    unit = str(other.get("granularity_type", "seconds")).strip().lower()
    if unit.endswith("s") is False:
        unit = unit + "s"
    return f"{n} {unit}"


def apply_startup_mask(section_raw: pd.Series, *, filter_value: float, startup_minutes: int) -> pd.Series:
    """
    Returns boolean mask (True = allowed) excluding startup period after CLOSED->OPEN transitions.
    """
    if section_raw is None or section_raw.empty:
        return pd.Series(False, index=pd.Index([]))

    sec = pd.to_numeric(section_raw, errors="coerce").sort_index()
    open_now = sec >= float(filter_value)

    transitions = (~open_now.shift(1).fillna(False)) & (open_now.fillna(False))
    allowed = pd.Series(True, index=sec.index)

    if startup_minutes and startup_minutes > 0 and transitions.any():
        for t in sec.index[transitions]:
            end = t + pd.Timedelta(minutes=int(startup_minutes))
            allowed.loc[(allowed.index >= t) & (allowed.index <= end)] = False

    return allowed

In [6]:
def pull_wide_runtime_parity(
    *,
    ssot: Dict[str, Any],
    sensor_name: str,
    start: datetime,
    end: datetime,
) -> Tuple[pd.DataFrame, pd.Series, Dict[str, Any]]:
    """
    Uses runtime build_wide_frame which already uses shutdown_rules.tags + shutdown_rules.shutdown_rules.
    Returns: df_wide, section_raw, sensor_cfg
    """
    sensor = get_sensor(ssot, sensor_name)
    df_wide, section_raw = build_wide_frame(db=db, sensor=sensor, start=start, end=end)
    return df_wide, section_raw, sensor.cfg

In [7]:
def train_statistical_thresholds(
    *,
    ssot: Dict[str, Any],
    sensor_name: str,
    start: datetime,
    end: datetime,
    mode: str = "quantile",          # "quantile" or "zscore"
    zscore_k: float = 3.0,
    quantile_hi: float = 0.99,
    quantile_lo: float = 0.01,
    resample_rule: str = "h",
    smoothing_window: int = 10,
    pull_interval: Optional[str] = None,  # if None, use SSOT granularity
) -> Dict[str, Any]:
    """
    Uses SSOT for:
      - startup_minutes (Other.startup_period)
      - filter_value (Other.filter_value)
      - shutdown_rules (via build_wide_frame)
    Uses widgets for:
      - mode, quantiles, zscore_k, resample_rule, smoothing_window, pull_interval
    Produces trained_thresholds.json-like dict.
    """
    df_wide, section_raw, sensor_cfg = pull_wide_runtime_parity(
        ssot=ssot, sensor_name=sensor_name, start=start, end=end
    )

    if df_wide is None or df_wide.empty:
        return {"ok": False, "error": "no_raw_data"}

    # Normalize index to UTC naive
    df_wide = df_wide.copy()
    df_wide.index = pd.to_datetime(df_wide.index, utc=True).tz_convert(None)
    df_wide = df_wide.sort_index()

    # Section series
    if section_raw is None or len(getattr(section_raw, "index", [])) == 0:
        section_raw = pd.Series(index=df_wide.index, data=np.nan, dtype=float)
    else:
        section_raw = section_raw.copy()
        section_raw.index = pd.to_datetime(section_raw.index, utc=True).tz_convert(None)
        section_raw = section_raw.sort_index().reindex(df_wide.index).ffill()

    other = (sensor_cfg.get("Other", {}) or {})
    startup_minutes = int(other.get("startup_period", 0))
    filter_value = float(other.get("filter_value", 0.9))

    # Gate open mask
    gate_open = (pd.to_numeric(section_raw, errors="coerce") >= filter_value).fillna(False)
    # Startup exclusion
    allowed = apply_startup_mask(section_raw, filter_value=filter_value, startup_minutes=startup_minutes)
    mask = gate_open & allowed
    df_use = df_wide.loc[mask].copy()

    # Resample (optional) on the used data
    if resample_rule:
        df_use = df_use.resample(resample_rule).mean()

    # Smoothing
    if smoothing_window and smoothing_window > 1:
        df_use = df_use.rolling(int(smoothing_window), min_periods=1).mean()

    # Determine feature tags from SSOT
    feature_displaynames_to_tags = {}
    for disp, meta in (sensor_cfg.get("Features", {}) or {}).items():
        if isinstance(meta, dict) and meta.get("tag"):
            feature_displaynames_to_tags[disp] = meta["tag"]

    tags = list(feature_displaynames_to_tags.values())
    if not tags:
        return {"ok": False, "error": "no_feature_tags"}

    # Compute thresholds per tag
    hi_map: Dict[str, float] = {}
    lo_map: Dict[str, float] = {}
    stats_map: Dict[str, Dict[str, Any]] = {}

    for tag in tags:
        if tag not in df_use.columns:
            continue
        s = pd.to_numeric(df_use[tag], errors="coerce").dropna()
        if s.empty:
            continue

        mu = float(s.mean())
        sd = float(s.std(ddof=0)) if float(s.std(ddof=0)) > 0 else 0.0
        n = int(s.shape[0])
        stats_map[tag] = {"mean": mu, "std": sd, "n": n}

        if mode == "zscore":
            hi = mu + float(zscore_k) * sd
            lo = mu - float(zscore_k) * sd
        else:
            hi = float(s.quantile(float(quantile_hi)))
            lo = float(s.quantile(float(quantile_lo)))

        hi_map[tag] = hi
        lo_map[tag] = lo

    trained_at_utc = datetime.utcnow().replace(microsecond=0).isoformat() + "Z"

    out = {
        "ok": True,
        "sensor": sensor_name,
        "method": "StatisticalThresholds",
        "trained_at_utc": trained_at_utc,
        "window": {
            "start": start.replace(microsecond=0).isoformat(),
            "end": end.replace(microsecond=0).isoformat(),
        },
        "training": {
            "mode": mode,
            "zscore_k": float(zscore_k),
            "quantile_hi": float(quantile_hi),
            "quantile_lo": float(quantile_lo),
            "resample_rule": resample_rule,
            "startup_minutes": int(startup_minutes),     # SSOT
            "filter_value": float(filter_value),        # SSOT
            "smoothing_window": int(smoothing_window),
            "pull_interval": pull_interval or interval_string_from_other(other),
        },
        "thresholds": {
            "high": hi_map,
            "low": lo_map,
            "stats": stats_map,
        },
    }
    return out

In [8]:
def trained_params_dir(site_root: Path, sensor_name: str, method: str) -> Path:
    out_dir = site_root / "etc" / "trained_params" / sensor_name / method
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir


def save_trained_thresholds(site_root: Path, trained: Dict[str, Any]) -> Path:
    sensor_name = trained["sensor"]
    method = trained["method"]
    out_dir = trained_params_dir(site_root, sensor_name, method)
    out_path = out_dir / "trained_thresholds.json"
    with open(out_path, "w") as f:
        json.dump(trained, f, indent=2)
    return out_path

In [9]:
def preview_thresholds(
    *,
    ssot: dict,
    sensor_name: str,
    trained: dict,
    db,
    n_tags: int = 6,
    lookback_hours: int = 48,
):
    sensor = get_sensor(ssot, sensor_name)
    sensor_cfg = sensor.cfg

    thr = trained.get("thresholds", {}) or {}
    hi_map = (thr.get("high", {}) or {})
    lo_map = (thr.get("low", {}) or {})

    tags = [t for t in hi_map.keys() if t in lo_map][:n_tags]
    if not tags:
        print("No thresholds to preview.")
        return

    end = dtparser.parse(trained["window"]["end"])
    start = end - timedelta(hours=int(lookback_hours))

    df_wide, section_raw = build_wide_frame(db=db, sensor=sensor, start=start, end=end)
    if df_wide is None or df_wide.empty:
        print("No plot data.")
        return

    df_wide = df_wide.copy()
    df_wide.index = pd.to_datetime(df_wide.index, utc=True).tz_convert(None)
    df_wide = df_wide.sort_index()

    section_raw = section_raw.copy() if section_raw is not None else pd.Series(index=df_wide.index, data=np.nan)
    section_raw.index = pd.to_datetime(section_raw.index, utc=True).tz_convert(None)
    section_raw = section_raw.sort_index().reindex(df_wide.index).ffill()

    other = (sensor_cfg.get("Other", {}) or {})
    filter_value = float(other.get("filter_value", 0.9))
    startup_minutes = int(other.get("startup_period", 0))

    gate_open = (pd.to_numeric(section_raw, errors="coerce") >= filter_value).fillna(False)
    allowed = apply_startup_mask(section_raw, filter_value=filter_value, startup_minutes=startup_minutes)
    mask = gate_open & allowed
    df_plot = df_wide.loc[mask].copy()

    fig, axes = plt.subplots(len(tags), 1, figsize=(14, 3 * len(tags)), sharex=True)
    if len(tags) == 1:
        axes = [axes]

    for ax, tag in zip(axes, tags):
        if tag not in df_plot.columns:
            ax.set_title(f"{tag} (missing)")
            continue
        s = pd.to_numeric(df_plot[tag], errors="coerce")
        ax.plot(s.index, s.values, label=tag)
        ax.axhline(float(hi_map[tag]), linestyle="--", label="High")
        ax.axhline(float(lo_map[tag]), linestyle="--", label="Low")
        ax.grid(True)
        ax.legend(loc="upper right")

    plt.show()

In [14]:
# --- DB "health check" + reconnect if needed ---
from src.apm_core.settings import load_ini
from src.apm_core.db_interface import DBInterface

def ensure_db_open():
    global db
    # If db doesn't exist or pg pool is closed, recreate it
    if "db" not in globals() or db is None:
        print("db not found -> creating new DBInterface")
    else:
        pool_closed = getattr(getattr(db, "pg_pool", None), "closed", True)
        if pool_closed:
            print("db.pg_pool is CLOSED -> recreating DBInterface")
        else:
            print("db.pg_pool is OPEN -> OK")
            return

    settings = load_ini(SITE_ROOT, debug=False)
    db = DBInterface(settings=settings, logger=logger)
    print("✅ db recreated")

ensure_db_open()

db.pg_pool is OPEN -> OK


In [12]:
sensor_dd = widgets.Dropdown(
    options=["ALL"] + SENSORS,
    value=SENSORS[0],
    description="Sensor:",
    layout=widgets.Layout(width="450px"),
)

start_txt = widgets.Text(
    value=(datetime.utcnow() - timedelta(days=120)).strftime("%Y-%m-%d %H:%M:%S"),
    description="Start:",
    layout=widgets.Layout(width="450px"),
)

end_txt = widgets.Text(
    value=datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
    description="End:",
    layout=widgets.Layout(width="450px"),
)

mode_dd = widgets.Dropdown(
    options=["quantile", "zscore"],
    value="quantile",
    description="Mode:",
    layout=widgets.Layout(width="250px"),
)

resample_dd = widgets.Dropdown(
    options=["", "min", "5min", "15min", "h", "4h", "D"],
    value="h",
    description="Resample:",
    layout=widgets.Layout(width="250px"),
)

q_hi = widgets.FloatText(value=0.99, description="q_hi:", layout=widgets.Layout(width="250px"))
q_lo = widgets.FloatText(value=0.01, description="q_lo:", layout=widgets.Layout(width="250px"))
z_k = widgets.FloatText(value=3.0, description="z_k:", layout=widgets.Layout(width="250px"))
smooth = widgets.IntSlider(value=10, min=1, max=60, step=1, description="Smooth:", layout=widgets.Layout(width="450px"))

pull_interval_txt = widgets.Text(
    value="",
    description="Pull interval:",
    placeholder="Leave blank to use SSOT granularity (e.g. '15 seconds')",
    layout=widgets.Layout(width="450px"),
)

btn = widgets.Button(description="Train Statistical Thr...", button_style="success")
out = widgets.Output()

def on_train_clicked(_):
    with out:
        clear_output()

        # Reconnect guard (prevents "pool is closed")
        global db
        pool_closed = getattr(getattr(db, "pg_pool", None), "closed", True)
        if pool_closed:
            settings = load_ini(SITE_ROOT, debug=False)
            db = DBInterface(settings=settings, logger=logger)

        selected = sensor_dd.value
        sensor_list = SENSORS if selected == "ALL" else [selected]

        start = dtparser.parse(start_txt.value)
        end = dtparser.parse(end_txt.value)

        mode = mode_dd.value
        resample_rule = resample_dd.value
        smoothing_window = int(smooth.value)
        pull_interval = pull_interval_txt.value.strip() or None

        for sensor_name in sensor_list:
            print("=" * 80)
            print("TRAINING:", sensor_name)

            trained = train_statistical_thresholds(
                ssot=ssot,
                sensor_name=sensor_name,
                start=start,
                end=end,
                mode=mode,
                zscore_k=float(z_k.value),
                quantile_hi=float(q_hi.value),
                quantile_lo=float(q_lo.value),
                resample_rule=resample_rule,
                smoothing_window=smoothing_window,
                pull_interval=pull_interval,
            )

            if not trained.get("ok", False):
                print("FAILED:", trained.get("error", "unknown_error"))
                continue

            out_path = save_trained_thresholds(SITE_ROOT, trained)
            print("✅ Saved ->", out_path)

        print("\nDone.")

        # quick preview
        preview_thresholds(ssot=ssot, sensor_name=sensor_name, trained=trained, db=db, n_tags=6)

btn.on_click(on_train_clicked)

display(widgets.VBox([
    sensor_dd,
    start_txt,
    end_txt,
    widgets.HBox([mode_dd, resample_dd]),
    widgets.HBox([q_hi, q_lo, z_k]),
    smooth,
    pull_interval_txt,
    btn,
    out
]))

In [15]:
db.close()
print("✅ DB connections closed")

✅ DB connections closed
